# L7 - Agent Config Versioning

Every change to an agent's system prompt is a change to its behaviour, so it is worth
treating those changes the way you treat code changes: snapshot what is running, propose a
revision, and keep an auditable chain between the two.

In this notebook you will:

- Read the Orchestrator Harness's current system prompt
- Ask Bedrock for a revised version, with a rationale
- Save both as versioned **configuration bundles** - a bundle is an immutable snapshot of the
  agent's settings, so the history works like a series of git commits
- Walk the version chain

Nothing here changes your running Harness. The candidate is recorded, not deployed.
Notebook 3 measures whether it is actually an improvement, and only then is promoting it
worth considering.

> **A note on AgentCore Optimization.** The AgentCore service can generate these
> recommendations for you, but it reads OpenTelemetry spans from CloudWatch. On the Datadog
> track those spans do not exist - the specialist agents run the Datadog trace-agent sidecar
> with ADOT disabled, and the Harness exports OTLP straight to Datadog. So this notebook asks
> Bedrock for the candidate directly. The AgentCore-driven version lives in the `aws-only`
> track, where ADOT is on.

## Steps

1. Setup - connect to AWS and read the current prompt
2. Ask Bedrock for a revised system prompt
3. Save the current config as bundle version 1
4. Save the candidate as bundle version 2
5. Walk the version history
6. Clean up


## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- **L3 notebook 4** - the Orchestrator Harness is deployed and `harness_id` / `harness_arn`
  are in SSM
- **L2 notebook 1** - `harness_model_id` is in SSM (the model used to draft the candidate)

**Cost note:** one Bedrock call. Well under a cent.


## Step 1: Setup

Install the Python packages we'll use. If they're already installed, this is a fast no-op.

In [ ]:
%pip install --quiet --upgrade 'boto3>=1.40.50' 'pandas==2.2.3'

Import libraries and set up connections to the AWS services we need. Load workshop settings that earlier notebooks (L1-L6) saved to AWS Systems Manager Parameter Store — like Harness ID and Cognito login info.

In [ ]:
import boto3
import json
import os

import pandas as pd

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# agentcore_control manages AgentCore resources: here, the Harness and its
# configuration bundles.
ssm = boto3.client("ssm", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

SSM_PREFIX = "/anycompany/agentcore"
ACCOUNT_ID = sts.get_caller_identity()["Account"]

# Load a few settings from Parameter Store
cfg = {}
for k in ["harness_id", "harness_arn", "harness_name", "harness_model_id"]:
    try:
        cfg[k] = ssm.get_parameter(Name="{}/{}".format(SSM_PREFIX, k), WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        cfg[k] = None
        print("  Warning: missing {}/{}".format(SSM_PREFIX, k))

if not cfg.get("harness_arn"):
    raise RuntimeError("Harness ARN not found in SSM. Run L3 notebook #4 first.")

print("Region:  {}".format(REGION))
print("Account: {}".format(ACCOUNT_ID))
print("Harness: {}".format(cfg["harness_arn"]))

Read the Harness's current settings so we know what we're trying to improve. This gives us its current system prompt so we can ask for a better one.

In [ ]:
resp = agentcore_control.get_harness(harnessId=cfg["harness_id"])
harness_state = resp["harness"]

# Flatten the system prompt (it comes back as a list of pieces)
CURRENT_SYSTEM_PROMPT = "\n".join(
    seg.get("text", "") for seg in harness_state.get("systemPrompt", []) if isinstance(seg, dict)
)
if not CURRENT_SYSTEM_PROMPT:
    raise RuntimeError("The Harness has no system prompt \u2014 nothing to optimize.")

print("Current system prompt: {} characters".format(len(CURRENT_SYSTEM_PROMPT)))
print(CURRENT_SYSTEM_PROMPT)

## Step 2: Ask Bedrock for a Revised System Prompt

We hand the model the prompt that is running today, plus the two failure modes this
orchestrator is known to hit - ambiguous requests where it should ask a clarifying question,
and multi-tool requests where it should gather everything before answering - and ask for a
revision.

The model returns the rewritten prompt and its reasoning. Treat both as a **proposal**: the
next notebook scores it against real data before anyone considers deploying it.


In [ ]:
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
MODEL_ID = cfg["harness_model_id"]

KNOWN_WEAKNESSES = """
1. Ambiguous requests: the agent guesses instead of asking one clarifying question.
2. Multi-tool requests: the agent answers after the first tool call instead of gathering
   everything it needs first.
"""

instruction = (
    "You improve system prompts for a customer-support orchestrator agent. The agent "
    "discovers specialist agents from a registry and delegates to them.\n\n"
    "Current system prompt:\n<prompt>\n" + CURRENT_SYSTEM_PROMPT + "\n</prompt>\n\n"
    "Known weaknesses:\n" + KNOWN_WEAKNESSES + "\n"
    "Rewrite the prompt to address those weaknesses. Preserve every existing capability, "
    "tool name, and delegation rule - this is a revision, not a rewrite from scratch.\n\n"
    "Reply with JSON only, no markdown fence:\n"
    '{"system_prompt": "<the full revised prompt>", '
    '"explanation": "<2-4 sentences on what you changed and why>"}'
)

resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": instruction}]}],
    inferenceConfig={"maxTokens": 4096, "temperature": 0.2},
)
raw = resp["output"]["message"]["content"][0]["text"].strip()

# The model is asked for bare JSON, but strip a fence if one appears anyway.
if raw.startswith("```"):
    raw = raw.split("```")[1]
    if raw.startswith("json"):
        raw = raw[4:]

RECOMMENDED_SYSTEM_PROMPT, sp_result = None, {}
try:
    parsed = json.loads(raw)
    RECOMMENDED_SYSTEM_PROMPT = (parsed.get("system_prompt") or "").strip()
    sp_result = {"explanation": parsed.get("explanation")}
except json.JSONDecodeError:
    print("Could not parse the response as JSON. Raw output below - copy the prompt into")
    print("RECOMMENDED_SYSTEM_PROMPT by hand, or re-run this cell.\n")
    print(raw[:2000])

if RECOMMENDED_SYSTEM_PROMPT:
    delta = len(RECOMMENDED_SYSTEM_PROMPT) - len(CURRENT_SYSTEM_PROMPT)
    print("Candidate prompt: {} characters ({:+d} vs current)".format(
        len(RECOMMENDED_SYSTEM_PROMPT), delta))

Let's compare the current prompt and the recommended one side by side, plus read the explanation of what changed and why.

In [ ]:
if not RECOMMENDED_SYSTEM_PROMPT:
    print("(No recommendation to show \u2014 see error above.)")
else:
    print("=" * 20 + " CURRENT PROMPT " + "=" * 20)
    print(CURRENT_SYSTEM_PROMPT[:1000])
    if len(CURRENT_SYSTEM_PROMPT) > 1000:
        print("... ({} chars total)".format(len(CURRENT_SYSTEM_PROMPT)))

    print("\n" + "=" * 20 + " RECOMMENDED PROMPT " + "=" * 20)
    print(RECOMMENDED_SYSTEM_PROMPT[:1000])
    if len(RECOMMENDED_SYSTEM_PROMPT) > 1000:
        print("... ({} chars total)".format(len(RECOMMENDED_SYSTEM_PROMPT)))

    print("\n" + "=" * 20 + " WHY THIS CHANGE " + "=" * 20)
    print(sp_result.get("explanation") or "(no explanation returned)")

## Step 3: Save the Current Config as a Bundle (Version 1)

A **configuration bundle** is a versioned, immutable snapshot of your agent's settings — system prompt, model, tools. Think of it like a git commit for your agent's configuration: every save creates a new frozen version you can go back to.

This step creates the bundle and saves the CURRENT Harness config as **version 1** on a branch called `mainline`. Nothing has changed about your live Harness — this is just a snapshot you can refer to later.

In [ ]:
BUNDLE_NAME = "anycompany_harness_config"

# Grab the Harness's current model config too, so the snapshot is complete.
harness_model = harness_state.get("model", {}).get("bedrockModelConfig", {}) or {}

# Convention: the bundle's component key is the ARN of the resource we're describing.
bundle_components = {
    cfg["harness_arn"]: {
        "configuration": {
            "systemPrompt": CURRENT_SYSTEM_PROMPT,
            "modelId": harness_model.get("modelId"),
            "temperature": harness_model.get("temperature"),
            "maxTokens": harness_model.get("maxTokens"),
        }
    }
}

try:
    resp = agentcore_control.create_configuration_bundle(
        bundleName=BUNDLE_NAME,
        description="AnyCompany orchestrator Harness configuration snapshot.",
        components=bundle_components,
        branchName="mainline",
        commitMessage="Initial snapshot of the deployed Harness configuration.",
    )
    BUNDLE_ID = resp["bundleId"]
    BUNDLE_ARN = resp["bundleArn"]
    INITIAL_VERSION_ID = resp["versionId"]
    print("Created bundle version 1:")
    print("  bundleId:  {}".format(BUNDLE_ID))
    print("  versionId: {}".format(INITIAL_VERSION_ID))
except agentcore_control.exceptions.ConflictException:
    BUNDLE_ID = None
    for b in agentcore_control.list_configuration_bundles().get("bundles", []):
        if b["bundleName"] == BUNDLE_NAME:
            BUNDLE_ID = b["bundleId"]
            BUNDLE_ARN = b["bundleArn"]
            break
    if not BUNDLE_ID:
        raise
    got = agentcore_control.get_configuration_bundle(bundleId=BUNDLE_ID)
    INITIAL_VERSION_ID = got["versionId"]
    print("Bundle already exists, reusing:")
    print("  bundleId:  {}".format(BUNDLE_ID))
    print("  latest versionId: {}".format(INITIAL_VERSION_ID))

## Step 4: Save the Candidate as Bundle Version 2

Version 1 holds the config that is running. Now save the candidate prompt as **version 2** of
the same bundle, recording version 1 as its parent so the chain reads like a commit history.

Version 1 is untouched and the live Harness is untouched. Version 2 is a recorded proposal -
notebook 2 decides whether it earns a deployment.


In [ ]:
if not RECOMMENDED_SYSTEM_PROMPT:
    print("Skipping \u2014 Step 2 did not produce a candidate prompt.")
    NEW_VERSION_ID = None
else:
    # Same components as version 1, with the candidate prompt swapped in.
    new_components = {
        cfg["harness_arn"]: {
            "configuration": {
                "systemPrompt": RECOMMENDED_SYSTEM_PROMPT,
                "modelId": harness_model.get("modelId"),
                "temperature": harness_model.get("temperature"),
                "maxTokens": harness_model.get("maxTokens"),
            }
        }
    }
    resp = agentcore_control.update_configuration_bundle(
        bundleId=BUNDLE_ID,
        components=new_components,
        parentVersionIds=[INITIAL_VERSION_ID],
        branchName="mainline",
        commitMessage="Candidate system prompt drafted in Step 2.",
    )
    NEW_VERSION_ID = resp["versionId"]
    print("Saved bundle version 2:")
    print("  versionId: {}".format(NEW_VERSION_ID))
    print("  parent:    {}".format(INITIAL_VERSION_ID))

## Step 5: Look at the Version History

List every version of the bundle to see the chain we just built. Each version records its parent, who created it, and a commit message — a full audit trail of your prompt changes.

In [ ]:
versions_resp = agentcore_control.list_configuration_bundle_versions(bundleId=BUNDLE_ID)
rows = []
for v in versions_resp.get("versions", []):
    lm = v.get("lineageMetadata", {})
    rows.append({
        "versionId": v["versionId"],
        "branch": lm.get("branchName"),
        "parents": ",".join(lm.get("parentVersionIds") or []) or "(root)",
        "author": (lm.get("createdBy") or {}).get("name") or "(system)",
        "message": (lm.get("commitMessage") or "")[:60],
        "createdAt": str(v.get("versionCreatedAt", ""))[:19],
    })

df = pd.DataFrame(rows)
print("Version history for bundle '{}':\n".format(BUNDLE_NAME))
print(df.to_string(index=False))

## Step 6: Clean Up

Delete the configuration bundle created here.

Leave it in place if you are moving on to notebook 2 - that notebook reads the candidate
prompt out of this bundle.


In [ ]:
## === CLEANUP: delete the configuration bundle created in this notebook ===
## Run this only once you are done with notebook 2, which reads the bundle.
# try:
#     agentcore_control.delete_configuration_bundle(bundleId=BUNDLE_ID)
#     print("Deleted configuration bundle {}".format(BUNDLE_ID))
# except Exception as e:
#     print("Bundle delete failed: {}".format(e))
print("Nothing deleted. Uncomment above once notebook 2 is done with the bundle.")